In [1]:
import sys
from pathlib import Path

# 1. Add the parent directory (src/) to sys.path
src_dir = Path.cwd().parent
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# 2. Use absolute imports (no leading dots)
from dotenv import load_dotenv
from ollama import ChatResponse

from helpers.config import get_settings
from stores.llm.LLMEnums import LLMEnums
from stores.llm.LLMProviderFactory import LLMProviderFactory

# 3. Run your test
load_dotenv()
config = get_settings()
factory = LLMProviderFactory(config=config)
client = factory.create(LLMEnums.OLLAMA.value)

client.set_generation_model(config.GENERATION_MODEL_ID)
response: ChatResponse = client.generate_response("Hi", None)

print(response.message.content)

Hello! I'm GLM, trained by Z.ai. How can I assist you today? Whether you have questions or just want to chat, I'm happy to help.


In [2]:
from stores.vectordb.VectorDBProviderFactory import VectorDBProviderFactory
from stores.vectordb.VectorDBEnums import VectorDBEnums

In [3]:
factory_db = VectorDBProviderFactory(config)

In [4]:
client_db = factory_db.create(provider=VectorDBEnums.QDRANT.value)

In [5]:
client_db.connect()

In [6]:
client_embed=factory.create(LLMEnums.OLLAMAE.value)

In [7]:
client_embed.set_embedding_model(config.EMBEDDING_MODEL_ID,config.EMBEDDING_MODEL_SIZE)

In [8]:
from tools.search_doctors import DoctorTools

doc_tool = DoctorTools(client_embed,client_db)
tool = doc_tool.search_doctors

In [9]:
from prompts.prompt_templatet import format_prompt,SYSTEM_PROMPT

In [10]:
query = "My leg is broken"

In [11]:
messages = [
    {
        "role":"system",
        "content":SYSTEM_PROMPT
    }
]

In [12]:
response =client.generate_response(prompt=query,
                                   chat_history=messages,
                                   tools=[tool])

In [13]:
response

ChatResponse(model='glm-5.2:cloud', created_at='2026-08-23T12:31:34.844092244Z', done=True, done_reason='stop', total_duration=5934429169, load_duration=None, prompt_eval_count=1028, prompt_eval_duration=None, eval_count=572, eval_duration=None, message=Message(role='assistant', content="I'm sorry to hear about your leg — that sounds painful and stressful. Let me help you right away.\n\n---\n\n### 🚨 Emergency Check\nIf the fracture just happened and you're experiencing any of the following, **please call 123 (Ambulance in Egypt) or go to the nearest Emergency Room immediately**:\n\n- Severe pain that's worsening\n- Visible deformity or bone protruding through the skin\n- Numbness, tingling, or loss of feeling in the leg/foot\n- The leg looks pale, blue, or cold to the touch\n- Heavy bleeding\n- You're unable to move or bear any weight at all\n\n**A fresh fracture is a medical emergency and should be assessed at an ER first** for immediate imaging (X-ray) and stabilization (splint/cast)

In [14]:
response.message.tool_calls

In [16]:
from agent.agent import Agent

agent = Agent(client,[tool],client_embedding=client_embed,
              client_vectorDB=client_db)

In [17]:
f,m,c=agent.run("my leg is broken")

In [18]:
print(f)

I'm really sorry to hear about your leg! 🙏 A broken leg is a serious injury that needs prompt medical attention.

---

## 🚨 Emergency First — Please Read Carefully

If this is a **recent injury** and you're experiencing any of the following, **please call 123 (Ambulance in Egypt) or go to the nearest Emergency Room immediately**:

- Visible deformity or bone protruding through the skin
- Severe, unbearable pain
- Numbness, tingling, or loss of feeling in the leg/foot
- The leg looks misaligned or bent at an abnormal angle
- Heavy bleeding
- Signs of shock (pale, cold, sweaty, rapid heartbeat, dizziness)
- Inability to move or bear any weight at all

**Do not attempt to realign the leg yourself.** Keep it as still as possible and immobilize it if you can while waiting for help.

---

## 🦴 For Follow-Up Orthopedic Care

Once you've received emergency treatment (or if this is a follow-up from a previous visit), you'll need to see an **Orthopedic Specialist** for proper evaluation, imaging

In [19]:
m

[{'role': 'system',
  'content': 'You are "Tabeeby" (طبيبي), an intelligent, empathetic, and professional AI Medical Assistant and Healthcare Navigator powered by the Vezeeta doctor network in Egypt.\n\n### YOUR ROLE & OBJECTIVES:\n1. **Medical Orientation & Guidance**: Listen carefully to the patient\'s symptoms or health inquiries, provide empathetic and evidence-based health information, and help determine the most suitable medical specialty (e.g., Cardiology, Dermatology, Orthopedics, Pediatrics, ENT, Internal Medicine, etc.).\n2. **Doctor Discovery & Recommendation**: Recommend the best-matching qualified doctors from your database/context based on the patient\'s symptoms, required specialty, location/area (e.g., Nasr City, Maadi, Dokki, Mohandessin, Heliopolis, Alexandria, etc.), consultation fee budget, and patient ratings.\n3. **Clear Next Steps**: Provide practical next steps, questions to ask the doctor, or general lifestyle/supportive measures.\n\n### CRITICAL MEDICAL GUARDR

In [20]:
c

0.003897

In [21]:
f,m,c = agent.run("can you search for me a doc in cairo and price be less than 400  that help me",
                  chat_history=m)

In [22]:
print(f)

I searched for orthopedic doctors in Cairo with a consultation fee under 400 EGP, but unfortunately **no matching doctors were found** within those exact criteria. 😔

But don't worry — let me help you find alternatives! Here are a few suggestions:

---

### 🔍 Let's Broaden the Search

Could you try any of the following to help me find the right doctor for you?

1. **Increase your budget slightly** — Sometimes raising the limit just a bit (e.g., 500 EGP) can open up more options.
2. **Specify a neighborhood in Cairo** — For example:
   - Nasr City
   - Maadi
   - Dokki
   - Mohandessin
   - Heliopolis
   - New Cairo
   - Shoubra
   - October City

   Searching a specific area may return more results.
3. **Remove the fee filter** — I can search for all orthopedic doctors in Cairo and you can pick the one that fits your budget.

---

### 🚨 Important Reminder

Since you mentioned your **leg is broken**, please remember:

- If this is a **fresh injury**, please go to the **nearest Emergency

In [ ]:
m

[{'role': 'system',
  'content': 'You are "Tabeeby" (طبيبي), an intelligent, empathetic, and professional AI Medical Assistant and Healthcare Navigator powered by the Vezeeta doctor network in Egypt.\n\n### YOUR ROLE & OBJECTIVES:\n1. **Medical Orientation & Guidance**: Listen carefully to the patient\'s symptoms or health inquiries, provide empathetic and evidence-based health information, and help determine the most suitable medical specialty (e.g., Cardiology, Dermatology, Orthopedics, Pediatrics, ENT, Internal Medicine, etc.).\n2. **Doctor Discovery & Recommendation**: Recommend the best-matching qualified doctors from your database/context based on the patient\'s symptoms, required specialty, location/area (e.g., Nasr City, Maadi, Dokki, Mohandessin, Heliopolis, Alexandria, etc.), consultation fee budget, and patient ratings.\n3. **Clear Next Steps**: Provide practical next steps, questions to ask the doctor, or general lifestyle/supportive measures.\n\n### CRITICAL MEDICAL GUARDR

In [23]:
c

0.0106114

In [24]:
f,m,c = agent.run("can you search for me a doc in new cairo and price be less than 400  that help me",
                  chat_history=m)

In [25]:
print(f)

Great news! I found **5 orthopedic doctors** in New Cairo with consultation fees within your budget (≤ 400 EGP). Here they are:

---

### 🦴 Doctor Recommendations — Orthopedics in New Cairo

---

#### 1. Dr. Ahmed Esmat
- **Specialty**: Orthopedics
- **Subspecialties**: Arthroscopy & Sports Injuries, Adult Spine Surgery, Bone Deformities, Foot & Ankle Orthopedics, Joint Replacement, Hand & Upper Limb Surgery, Pediatric Orthopedics, and more
- **Clinic Location**: New Cairo
- **Consultation Fee**: 300 EGP
- **Reviews**: No reviews yet
- **Profile & Booking**: [View Profile](https://www.vezeeta.com/en/dr/doctor-ahmed-esmat-orthopedics)
- **Why this doctor**: Dr. Ahmed has a very wide range of orthopedic subspecialties, including **Foot & Ankle Orthopedics** and **Bone Deformities**, which are highly relevant to a broken leg. Great value at 300 EGP.

---

#### 2. Dr. Ramy Al Saeid Ahmed
- **Specialty**: Orthopedics
- **Subspecialties**: Adult Orthopedics
- **Clinic Location**: New Cairo
-

In [26]:
m

[{'role': 'system',
  'content': 'You are "Tabeeby" (طبيبي), an intelligent, empathetic, and professional AI Medical Assistant and Healthcare Navigator powered by the Vezeeta doctor network in Egypt.\n\n### YOUR ROLE & OBJECTIVES:\n1. **Medical Orientation & Guidance**: Listen carefully to the patient\'s symptoms or health inquiries, provide empathetic and evidence-based health information, and help determine the most suitable medical specialty (e.g., Cardiology, Dermatology, Orthopedics, Pediatrics, ENT, Internal Medicine, etc.).\n2. **Doctor Discovery & Recommendation**: Recommend the best-matching qualified doctors from your database/context based on the patient\'s symptoms, required specialty, location/area (e.g., Nasr City, Maadi, Dokki, Mohandessin, Heliopolis, Alexandria, etc.), consultation fee budget, and patient ratings.\n3. **Clear Next Steps**: Provide practical next steps, questions to ask the doctor, or general lifestyle/supportive measures.\n\n### CRITICAL MEDICAL GUARDR

In [27]:
c

0.0239826

In [28]:
f,m,c = agent.run("less than 400 and not eqal 400 also",
                  chat_history=m)

In [29]:
print(f)

Got it! You want doctors with a fee **strictly less than 400 EGP**. Let me filter out the 400 EGP doctors for you. Here are the ones that match:

---

### 🦴 Orthopedic Doctors in New Cairo — Fee Under 400 EGP

---

#### 1. Dr. Ahmed Esmat
- **Specialty**: Orthopedics
- **Subspecialties**: Arthroscopy & Sports Injuries, Adult Spine Surgery, Bone Deformities, Foot & Ankle Orthopedics, Joint Replacement, Hand & Upper Limb Surgery, Pediatric Orthopedics, and more
- **Clinic Location**: New Cairo
- **Consultation Fee**: **300 EGP**
- **Reviews**: No reviews yet
- **Profile & Booking**: [View Profile](https://www.vezeeta.com/en/dr/doctor-ahmed-esmat-orthopedics)
- **Why this doctor**: Very broad orthopedic expertise including **Foot & Ankle Orthopedics** and **Bone Deformities** — highly relevant to a broken leg.

---

#### 2. Dr. AbdElsamie Halawa ⭐ (Top Pick)
- **Specialty**: Orthopedics
- **Subspecialties**: Adult Orthopedics, Arthroscopy & Sports Injuries, Foot & Ankle Orthopedics, Joint

In [30]:
m

[{'role': 'system',
  'content': 'You are "Tabeeby" (طبيبي), an intelligent, empathetic, and professional AI Medical Assistant and Healthcare Navigator powered by the Vezeeta doctor network in Egypt.\n\n### YOUR ROLE & OBJECTIVES:\n1. **Medical Orientation & Guidance**: Listen carefully to the patient\'s symptoms or health inquiries, provide empathetic and evidence-based health information, and help determine the most suitable medical specialty (e.g., Cardiology, Dermatology, Orthopedics, Pediatrics, ENT, Internal Medicine, etc.).\n2. **Doctor Discovery & Recommendation**: Recommend the best-matching qualified doctors from your database/context based on the patient\'s symptoms, required specialty, location/area (e.g., Nasr City, Maadi, Dokki, Mohandessin, Heliopolis, Alexandria, etc.), consultation fee budget, and patient ratings.\n3. **Clear Next Steps**: Provide practical next steps, questions to ask the doctor, or general lifestyle/supportive measures.\n\n### CRITICAL MEDICAL GUARDR

In [31]:
c

0.034136